# __CH09 백터 저장소(Vector Store)__
<hr>

+ __벡터스토어 저장의 필요성__
  - 빠른 검색 속도
  - 스케일러빌리티
  - 의미 검색(Semantic Search) 지원
<br><br>
+ __벡터스토어 중요성__
  - RAG 시스템의 검색 기능과 직접적으로 연결
  - 전체 시스템의 응답 시간과 정확성에 큰 영향을 미침

### __1. Chroma__

#### __(1) Chroma__

* 개발자의 생산성과 행복에 초점을 맞춘 AI 네이티브 오픈 소스 벡터 데이터베이스
* Apache 2.0에 따라 라이선스가 부여
* 참조
  > [Chroma 공식 사이트](https://docs.trychroma.com/docs/overview/introduction)

In [ ]:
###################################
# 환경변수 정보 조회
###################################

# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TextSplitter 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)

# 텍스트 파일을 load -> List[Document] 형태로 변환
loader1 = TextLoader("./data/nlp-keywords.txt", encoding="UTF-8")
loader2 = TextLoader("./data/finance-keywords.txt", encoding="UTF-8")

# 문서 분할
split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)

# 문서 개수 확인
len(split_doc1), len(split_doc2)

#### __(2) VectorStore 생성__

##### __2-1) 벡터 저장소 생성 (from_documents)__
  * __`from_documents`__ 클래스 메서드는 문서 리스트로부터 벡터 저장소를 생성합니다.

__매개변수__
  * __`documents`__ (List[Document]): 벡터 저장소에 추가할 문서 리스트
  * __`embedding`__ (Optional[Embeddings]): 임베딩 함수. 기본값은 None
  * __`ids`__ (Optional[List[str]]): 문서 ID 리스트. 기본값은 None
  * __`collection_name`__ (str): 생성할 컬렉션 이름.
  * __`persist_directory`__ (Optional[str]): 컬렉션을 저장할 디렉토리. 기본값은 None
  * __`client_settings`__ (Optional[chromadb.config.Settings]): Chroma 클라이언트 설정
  * __`client`__ (Optional[chromadb.Client]): Chroma 클라이언트 인스턴스
  * __`collection_metadata`__ (Optional[Dict]): 컬렉션 구성 정보. 기본값은 None

__참고__
  * persist_directory가 지정되면 컬렉션이 해당 디렉토리에 저장됩니다. 지정되지 않으면 데이터는 메모리에 임시로 저장됩니다.
  * 이 메서드는 내부적으로 from_texts 메서드를 호출하여 벡터 저장소를 생성합니다.
  * 문서의 page_content는 텍스트로, metadata는 메타데이터로 사용됩니다.
    
__반환값__
  * __`Chroma`__: 생성된 Chroma 벡터 저장소 인스턴스 생성시 documents 매개변수로 Document 리스트를 전달합니다.<br>
  embedding 에 활용할 임베딩 모델을 지정하며, namespace 의 역할을 하는 collection_name 을 지정할 수 있습니다.

In [18]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

ollama_embeddings = OllamaEmbeddings(
    #model="nomic-embed-text",
    model="embeddinggemma",    
)

# 저장할 경로 지정
DB_PATH = "./chroma_db"

# DB 생성
db = Chroma.from_documents(
    documents=split_doc1, 
    embedding=ollama_embeddings,
    persist_directory=DB_PATH,
    collection_name="my_db"
)

# 디스크에서 문서를 로드합니다.
persist_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=ollama_embeddings,
    collection_name="my_db",
)

# 저장된 데이터 확인
persist_db.get()

{'ids': ['6097bc6e-b0df-4d8f-89f0-e038e52177b6',
  '980e2cc9-24f6-44e8-9f50-9b82fedb749d',
  'db356a6d-6cd0-4f24-ab40-31d671ccf856',
  'e4b17301-83bd-4c4a-84bd-0d2bca330ad8',
  '0329b772-4ce1-4e22-bc7a-88d827844dad',
  '00c945e5-580c-4b21-851d-52e898d80eaf',
  'effbdba4-5a05-4018-83ac-d708504f22b1',
  '6e1a33a7-2555-42e7-a65e-2a5e02f92504',
  '34a529a8-0571-42e1-af21-048ffb4c3b67',
  '9e51d8fe-19c1-4a35-8abb-2fbdc020a599',
  'bf6e9a4d-c263-4f1f-ba7b-e3396a31fdff',
  '2ebf9a08-13f4-49d3-b62e-5e5d68df0a17',
  '34bc93b3-a163-429a-9126-75a8a8580490',
  '85b0cb51-37ea-4dea-8968-79975248cc63',
  '1000a528-691d-4457-9d45-a59268297113',
  '2b1f4d25-90c4-48d6-8e13-a41147782118',
  '2bf39cad-2b31-4c1b-86d8-bdde5b36dac2',
  '6225714f-554e-4ea2-bac7-342b38423b16',
  '36f22b31-422d-4de2-9775-83aefa0b29e6',
  '14134abf-41c7-44ee-b467-c15aca76e790',
  'fedd2d90-a898-4f00-bde0-0365ff766f45',
  '52485e4c-9ef7-4f8b-bc44-4595899a73dd',
  '5d61118b-6edf-4a3b-ad77-aed5ae07581c',
  '17a8fe5a-ca90-408f-87fe-

##### __2-2) 벡터 저장소 생성 (from_texts)__

__`from_texts`__ 클래스 메서드는 텍스트 리스트로부터 벡터 저장소를 생성합니다.

__매개변수__
  * __`texts`__ (List[str]): 컬렉션에 추가할 텍스트 리스트
  * __`embedding`__ (Optional[Embeddings]): 임베딩 함수. 기본값은 None
  * __`metadatas`__ (Optional[List[dict]]): 메타데이터 리스트. 기본값은 None
  * __`ids`__ (Optional[List[str]]): 문서 ID 리스트. 기본값은 None
  * __`collection_name`__ (str): 생성할 컬렉션 이름. 기본값은 '_LANGCHAIN_DEFAULT_COLLECTION_NAME'
  * __`persist_directory`__ (Optional[str]): 컬렉션을 저장할 디렉토리. 기본값은 None
  * __`client_settings`__ (Optional[chromadb.config.Settings]): Chroma 클라이언트 설정
  * __`client`__ (Optional[chromadb.Client]): Chroma 클라이언트 인스턴스
  * __`collection_metadata`__ (Optional[Dict]): 컬렉션 구성 정보. 기본값은 None

__참고__
  * __`persist_directory`__가 지정되면 컬렉션이 해당 디렉토리에 저장됩니다. 지정되지 않으면 데이터는 메모리에 임시로 저장됩니다.
ids가 제공되지 않으면 UUID를 사용하여 자동으로 생성됩니다.

__반환값__

생성된 벡터 저장소 인스턴스

In [ ]:
# 문자열 리스트로 생성
db2 = Chroma.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 Froggy입니다."],
    embedding=OpenAIEmbeddings(),
)